# 01 — Prototype: known stars end to end

The first gate in the build order (handover, *Build order* step 2; spec §11): for every star
in the fixture manifest, run clean → periodograms → aliases → blend → Agreement → CMD →
plots, and compare the derived period with the published one. `tests/test_prototype.py`
runs the same pipeline on the same fixtures and enforces the pass criterion
(`argelander.prototype.judge`); this notebook is where the results are looked at.

Everything here is imported from `argelander`; nothing scientific is defined in the notebook.

**Sources.** By default the notebook reads the trimmed fixtures (`tests/fixtures/lightcurves/`,
at most 500 points per band) and the recorded Gaia catalog join (`tests/fixtures/catalog/`).
Set `LIVE = True` to fetch every star from the archives instead, untrimmed, into
`data/prototype-cache/`; that takes tens of minutes. A survey that fails or has no data is a
coverage gap, not an error: ASAS-SN is one until Sky Patrol is back (NEU-1764, NEU-1796).

Types are not proposed yet (the classifier is M4); the table shows Gaia's own class beside
the published type.

In [ ]:
import asyncio
from pathlib import Path

import yaml
from astropy.table import Table
from IPython.display import Image, Markdown, display

from argelander.analysis.pipeline import analyze
from argelander.catalog.gaia import fetch_catalog, records_from_tables
from argelander.config import get_settings
from argelander.fetch.asassn import AsassnFetcher
from argelander.fetch.base import ArchiveLimits, TargetRef
from argelander.fetch.gaia_epoch import GaiaEpochFetcher
from argelander.fetch.tess import TessFetcher
from argelander.fetch.ztf import ZtfFetcher
from argelander.lightcurve import cache_path, load_target
from argelander.plots import plot_dir, render_all
from argelander.prototype import judge

LIVE = False

REPO = Path("..").resolve()
FIXTURES = REPO / "tests" / "fixtures"
STARS = yaml.safe_load((FIXTURES / "manifest.yaml").read_text())
PLOTS = REPO / "data" / "prototype-plots"

## Light Curves and the Gaia catalog join

Fixtures by default. The live path fetches the catalog join first, because the TESS fetcher
needs each star's 120″ field, then every survey at once under the archives' semaphores, as
`scripts/build_fixtures.py` does, writing untrimmed Light Curves to the Cache layout.

In [ ]:
async def fetch_live(stars: list[dict], cache_dir: Path) -> dict:
    settings = get_settings()
    limits = ArchiveLimits.from_settings(settings)
    attempts = settings.fetch_max_attempts
    records = await fetch_catalog([s["gaia_source_id"] for s in stars], max_attempts=attempts)
    targets = [
        TargetRef(
            gaia_source_id=s["gaia_source_id"],
            ra=s["ra"],
            dec=s["dec"],
            vsx_name=s["star"],
            tess_field=records[s["gaia_source_id"]].tess_field,
        )
        for s in stars
    ]
    fetchers = [f(limits, max_attempts=attempts) for f in (ZtfFetcher, TessFetcher, AsassnFetcher)]

    def write(survey: str, target: TargetRef, curves: list) -> None:
        for lc in curves:
            lc.to_parquet(cache_path(cache_dir, survey, target.gaia_source_id, lc.band))

    async def one(fetcher, target: TargetRef) -> None:
        try:
            write(fetcher.survey, target, await fetcher.fetch(target))
        except Exception as error:  # a coverage gap, reported and skipped
            print(f"{target.vsx_name} {fetcher.survey}: {error!r}")

    async def gaia() -> None:
        try:
            curves = await GaiaEpochFetcher(limits, max_attempts=attempts).fetch_many(targets)
        except Exception as error:
            print(f"gaia epoch photometry: {error!r}")
            return
        for target in targets:
            write("gaia", target, curves[target.gaia_source_id])

    await asyncio.gather(gaia(), *(one(f, t) for f in fetchers for t in targets))
    return records


if LIVE:
    CACHE = REPO / "data" / "prototype-cache"
    RECORDS = await fetch_live(STARS, CACHE)
else:
    CACHE = FIXTURES / "lightcurves"
    RECORDS = records_from_tables(
        *(
            Table.read(FIXTURES / "catalog" / f"{name}.xml", format="votable")
            for name in ("catalog", "cross_ids", "neighbours", "tess_field")
        )
    )

## Analysis and plots, star by star

Most of a minute a star on the trimmed fixtures, nearly all of it BoxLeastSquares on the long
ZTF and Gaia baselines. Plots go to `data/prototype-plots/{gaia_source_id}/{pipeline_version}/`.

In [ ]:
RESULTS = {}
for star in STARS:
    gaia_source_id = star["gaia_source_id"]
    analysis, cleaned = analyze(
        load_target(CACHE, gaia_source_id), RECORDS.get(gaia_source_id), target_id=gaia_source_id
    )
    period = analysis.agreement.agreed_period if analysis.agreement else None
    plots = render_all(
        analysis,
        cleaned,
        vsx_type=star["published_type"] or "VAR",
        period=period,
        epoch=None,
        out_dir=plot_dir(PLOTS, gaia_source_id, analysis.pipeline_version),
    )
    RESULTS[star["star"]] = (analysis, cleaned, plots, judge(star, analysis))
    print(f"{star['star']}: {RESULTS[star['star']][3].reason}")

## Summary: derived vs published

One row per star. *Votes* are the sources' alias-tested best periods, as `agreement.votes`
casts them: a band whose best fold explains too little does not vote
(`agreement.MAX_SCATTER`), and one marked ½ is a survey the blend checks found contaminated,
counting at half weight. *Relation* is how the agreed period stands to the published one,
within the Agreement's tolerance.

In [ ]:
from argelander.analysis.agreement import SURVEYS
from argelander.analysis.agreement import votes as cast_votes


def votes(analysis, record) -> str:
    contaminated = analysis.blend.surveys if analysis.blend else frozenset()
    gaia_period = record.vari_period if record else None
    return ", ".join(
        f"{v.source} {v.period:.6g}{' ½' if SURVEYS[v.source] in contaminated else ''}"
        for v in cast_votes(analysis.alias_tests, gaia_period)
    )


def fmt(value, spec=".7g") -> str:
    return "–" if value is None else format(value, spec)


rows = [
    "| Star | Role | Published P (d) | Agreed P (d) | Relation | Sources | Votes | "
    "Published type | Gaia class | Flags | CMD | Gate |",
    "| --- | --- | ---: | ---: | --- | --- | --- | --- | --- | --- | --- | --- |",
]
for star in STARS:
    analysis, _, _, result = RESULTS[star["star"]]
    agreement = analysis.agreement
    record = RECORDS.get(star["gaia_source_id"])
    role = star["role"] if star["role"] == "positive" else star["negative_kind"]
    rows.append(
        f"| {star['star']} | {role} | {fmt(star['published_period'])} "
        f"| {fmt(agreement.agreed_period if agreement else None)} | {result.relation or '–'} "
        f"| {', '.join(agreement.agreed_sources) if agreement else '–'} "
        f"| {votes(analysis, record)} "
        f"| {star['published_type'] or '–'} | {(record.vari_class if record else None) or '–'} "
        f"| {', '.join(sorted(analysis.blend.flags)) if analysis.blend else '–'} "
        f"| {analysis.cmd.region if analysis.cmd else '–'} "
        f"| {'pass' if result.passed else '**FAIL**'} |"
    )
passed = sum(r[3].passed for r in RESULTS.values())
rows.append(f"\n**{passed} of {len(RESULTS)} stars pass the gate.**")
display(Markdown("\n".join(rows)))

## Evidence for one star

`submission.png` is the composite the VSX revision form takes: the phase plot of the band
with the most points above the combined light curve. Change `STAR` to look at another.

In [ ]:
STAR = "V0364 Dra"
_, _, plots, result = RESULTS[STAR]
print(result.reason)
for kind in ("submission", "periodogram", "cmd"):
    display(Image(filename=plots[kind]))